# v7 — tracksdata GLOBAL ILP linking (Phase-2 paradigm, step 2)

Offline ILP gate PASSED (`util_tracksdata_offline_test.ipynb`). **Step 2 = swap ONLY the linker**: keep our v1 UNet
detector (HM_THR 0.3, min_dist 3, CoM refine — identical to the v6 A/B), replace the frame-to-frame two-pass Hungarian
(v2) with royerlab **`tracksdata`'s global SCIP ILP**. Clean single-variable A/B on the last-5 `6bba_` val: v2 greedy
vs global ILP, same detections, same `edge_jaccard`/`division_jaccard` scorers.

- **Graph:** each detection = a node (t,z,y,x). Candidate edges between consecutive frames within `LINK_GATE_UM` in
  **physical µm** (cKDTree), capped at `MAX_NEIGH` per node. Edge weight `w = dist_um - EDGE_BIAS_UM` (short links → reward).
- **ILP:** `ILPSolver(edge_weight='w', appearance/disappearance/division weights)`, SCIP backend (Gurobi auto-falls-back).
- **First run:** `DIVISION_W=1000` → effectively strict 1:1, to isolate *global vs greedy* on edge-J. Lower it later to
  turn on divisions (watch `nPredDiv` for a trackastra-style flood).

**Run:** internet OFF; attach **competition data** + **`biohub-v1-unet-base16-heatmap`** (v1 weights) + **`tracksdata-offline`**
(wheels). Installs tracksdata `--no-deps` from the wheels (keeps Kaggle numpy so torch is unaffected), runs the A/B in a
fresh subprocess. **Read: ILP edge-J ≥ v2 0.8594 → global linking wins → the paradigm has legs.**


In [1]:
# install tracksdata + SCIP from the offline wheels (--no-deps keeps Kaggle numpy/scipy so torch is unaffected)
import os, sys, subprocess
from pathlib import Path

CANDS = [Path('/kaggle/input/tracksdata-offline/wheels'), Path('/kaggle/input/tracksdata-offline')]
def find_wheels():
    for c in CANDS:
        if c.exists() and list(c.glob('*.whl')): return c
    inp = Path('/kaggle/input')
    hits = list(inp.glob('**/tracksdata-*.whl')) + list(inp.glob('**/tracksdata_*.whl'))
    if hits: return hits[0].parent
    from collections import Counter
    d = Counter(w.parent for w in inp.glob('**/*.whl'))
    return d.most_common(1)[0][0] if d else None

wdir = find_wheels()
print('wheels dir:', wdir)
assert wdir is not None, 'attach the tracksdata-offline dataset (wheels)'
wheel_files = sorted(str(w) for w in wdir.glob('*.whl'))
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps'] + wheel_files,
                   capture_output=True, text=True)
print(r.stdout[-800:]); print('--- stderr tail ---'); print(r.stderr[-600:]); print('install rc', r.returncode)


wheels dir: /kaggle/input/datasets/lingxd/tracksdata-offline/wheels
stall: blosc2
    Found existing installation: blosc2 4.1.2
    Uninstalling blosc2-4.1.2:
      Successfully uninstalled blosc2-4.1.2

--- stderr tail ---

install rc 0


## A/B eval script (v1 detector + v2 baseline + tracksdata ILP), run in a fresh subprocess

Everything heavy runs in one subprocess so numpy stays consistent post-install. Detection + v2 linking + scorers are
identical to the v6 A/B; only `tracksdata_ilp_link` is new.


In [2]:
%%writefile /tmp/v7_ilp_eval.py

import os, glob, sys, time
os.environ.setdefault('POLARS_PREFER_PKG', '32')
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from skimage.feature import peak_local_max
import zarr

# ---------------- config (detection identical to v2; only linking differs) ----------------
VOXEL   = np.array([1.625, 0.40625, 0.40625])
NORM_PCT = (50.0, 99.8)
HM_THR   = 0.3
MIN_DIST = 3
REFINE_WIN = (1, 3, 3)
MATCH_UM = 7.0
TIGHT_UM = 6.0; LOOSE_UM = 8.0; VEL_BLEND = 0.5
MAX_GAP = 1; GAP_DIST_UM = 6.0; FILTER_MIN_LEN = 4
N_VAL    = 5
BASE     = 16
STRIDES  = ((1,2,2),(1,2,2),(2,2,2),(2,2,2))
INPUT    = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TRAIN_DIR = os.path.join(INPUT, 'train')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', device, '| numpy', np.__version__, '| torch', torch.__version__)

# ---------------- IO ----------------
def open_image(zp):
    g = zarr.open(zp, mode='r')
    if hasattr(g, 'shape'): return g
    best, bs = None, -1
    def walk(n):
        nonlocal best, bs
        try:
            for k in n.keys():
                s = n[k]
                if hasattr(s, 'shape'):
                    z = int(np.prod(s.shape))
                    if z > bs: best, bs = s, z
                else: walk(s)
        except Exception: pass
    walk(g); return best

def load_geff(gp):
    g = zarr.open(gp, mode='r')
    ids = np.asarray(g['nodes/ids'][:])
    props = {k: np.asarray(g['nodes/props/'+k+'/values'][:]) for k in g['nodes/props'].keys()}
    try: edges = np.asarray(g['edges/ids'][:])
    except Exception: edges = np.zeros((0,2))
    return ids, props, edges

def gt_nodes_edges(gp):
    ids, props, edges = load_geff(gp)
    nodes = [(int(ids[i]), int(props['t'][i]), int(props['z'][i]), int(props['y'][i]), int(props['x'][i]))
             for i in range(len(ids))]
    return nodes, [(int(s), int(t)) for s, t in edges]

def sample_list(d):
    out = []
    for gp in sorted(glob.glob(os.path.join(d, '*.geff'))):
        zp = gp[:-5] + '.zarr'
        if os.path.exists(zp): out.append((zp, gp))
    return out

# ---------------- detector (v1 arch, BASE16 InstanceNorm, anisotropic) ----------------
class ConvBlock(nn.Module):
    def __init__(s, ci, co):
        super().__init__()
        s.net = nn.Sequential(
            nn.Conv3d(ci, co, 3, padding=1, bias=False), nn.InstanceNorm3d(co, affine=True), nn.LeakyReLU(0.01, True),
            nn.Conv3d(co, co, 3, padding=1, bias=False), nn.InstanceNorm3d(co, affine=True), nn.LeakyReLU(0.01, True))
    def forward(s, x): return s.net(x)

class UNet3D(nn.Module):
    def __init__(s, in_ch=1, base=BASE, strides=STRIDES):
        super().__init__()
        n = len(strides); chs = [base*(2**i) for i in range(n+1)]
        s.enc, s.downs = nn.ModuleList(), nn.ModuleList(); cin = in_ch
        for i in range(n):
            s.enc.append(ConvBlock(cin, chs[i])); s.downs.append(nn.Conv3d(chs[i], chs[i], strides[i], strides[i])); cin = chs[i]
        s.bottleneck = ConvBlock(chs[n-1], chs[n])
        s.ups, s.dec = nn.ModuleList(), nn.ModuleList()
        for i in reversed(range(n)):
            s.ups.append(nn.ConvTranspose3d(chs[i+1], chs[i], strides[i], strides[i])); s.dec.append(ConvBlock(chs[i]*2, chs[i]))
        s.head = nn.Conv3d(chs[0], 1, 1)
    def forward(s, x):
        sk = []
        for e, d in zip(s.enc, s.downs): x = e(x); sk.append(x); x = d(x)
        x = s.bottleneck(x)
        for u, dec, k in zip(s.ups, s.dec, reversed(sk)):
            x = u(x)
            if x.shape[2:] != k.shape[2:]: x = F.interpolate(x, size=k.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, k], 1))
        return s.head(x)

def _norm(vol):
    v = vol.astype(np.float32); lo, hi = np.percentile(v, NORM_PCT)
    return np.clip((v-lo)/(hi-lo+1e-6), 0, 1).astype(np.float32)

@torch.no_grad()
def predict_hm(model, vol):
    v = _norm(vol); x = torch.from_numpy(np.ascontiguousarray(v)[None, None]).float().to(device)
    with torch.amp.autocast(device, enabled=(device == 'cuda')): y = torch.sigmoid(model(x))
    return y[0, 0].float().cpu().numpy()

def refine(vol, coords, win=REFINE_WIN):
    if len(coords) == 0: return coords
    Z, Y, X = vol.shape; out = coords.copy().astype(np.float64); wz, wy, wx = win
    for i, (z, y, x) in enumerate(coords):
        z, y, x = int(round(z)), int(round(y)), int(round(x))
        z0, z1 = max(0, z-wz), min(Z, z+wz+1); y0, y1 = max(0, y-wy), min(Y, y+wy+1); x0, x1 = max(0, x-wx), min(X, x+wx+1)
        p = vol[z0:z1, y0:y1, x0:x1].astype(np.float64); ss = p.sum()
        if ss <= 0: continue
        zz = np.arange(z0, z1)[:, None, None]; yy = np.arange(y0, y1)[None, :, None]; xx = np.arange(x0, x1)[None, None, :]
        out[i, 0] = (p*zz).sum()/ss; out[i, 1] = (p*yy).sum()/ss; out[i, 2] = (p*xx).sum()/ss
    return out

def detect(model, arr):
    frames = []
    for t in range(arr.shape[0]):
        vol = np.asarray(arr[t]).astype(np.float32); hm = predict_hm(model, vol)
        pk = peak_local_max(hm, min_distance=MIN_DIST, threshold_abs=HM_THR, exclude_border=False, num_peaks=200000)
        frames.append(refine(vol, pk.astype(np.float64)) if len(pk) else np.zeros((0, 3)))
    return frames

# ---------------- scorer (identical to repo edge_jaccard) ----------------
def match_nodes(gt_nodes, pred_nodes, max_um=MATCH_UM):
    gt_by, pr_by = defaultdict(list), defaultdict(list)
    for n in gt_nodes: gt_by[n[1]].append(n)
    for n in pred_nodes: pr_by[n[1]].append(n)
    g2p = {}
    for t, G in gt_by.items():
        P = pr_by.get(t, [])
        if not P: continue
        A = np.array([[g[2], g[3], g[4]] for g in G])*VOXEL
        B = np.array([[p[2], p[3], p[4]] for p in P])*VOXEL
        D = cdist(A, B); r, c = linear_sum_assignment(D)
        for i, j in zip(r, c):
            if D[i, j] <= max_um: g2p[G[i][0]] = P[j][0]
    return g2p

def edge_jaccard(gt_nodes, gt_edges, pred_nodes, pred_edges):
    g2p = match_nodes(gt_nodes, pred_nodes); p2g = {v: k for k, v in g2p.items()}
    pred_set = set(map(tuple, pred_edges)); gt_set = set(map(tuple, gt_edges))
    TP = FP = 0
    for a, b in pred_edges:
        if a in p2g and b in p2g:
            TP += (p2g[a], p2g[b]) in gt_set; FP += (p2g[a], p2g[b]) not in gt_set
    FN = sum(1 for u, v in gt_edges if not (u in g2p and v in g2p and (g2p[u], g2p[v]) in pred_set))
    d = TP+FP+FN
    return dict(J=TP/d if d else 0.0, TP=TP, FP=FP, FN=FN), g2p

def division_jaccard(gt_nodes, gt_edges, pred_nodes, pred_edges, g2p=None):
    if g2p is None: g2p = match_nodes(gt_nodes, pred_nodes)
    p2g = {v: k for k, v in g2p.items()}
    def outdeg(edges):
        d = defaultdict(int)
        for a, b in edges: d[a] += 1
        return d
    gd, pd = outdeg(gt_edges), outdeg(pred_edges)
    gt_div = set(u for u, c in gd.items() if c >= 2)
    pred_div = set(u for u, c in pd.items() if c >= 2)
    TP = sum(1 for u in gt_div if u in g2p and g2p[u] in pred_div)
    FN = len(gt_div) - TP
    FP = sum(1 for p in pred_div if (p2g.get(p) is None or p2g.get(p) not in gt_div))
    d = TP+FP+FN
    return dict(divJ=TP/d if d else 0.0, TP=TP, FP=FP, FN=FN, n_gt_div=len(gt_div), n_pred_div=len(pred_div))

# ---------------- v2 two-pass linking (baseline) ----------------
def build_nodes(frames):
    nodes = []; fid = []; nid = 1
    for t, coords in enumerate(frames):
        ids = []
        for (z, y, x) in coords:
            nodes.append((nid, t, z, y, x)); ids.append(nid); nid += 1
        fid.append(ids)
    return nodes, fid

def link_twopass(frames, fid, tight_um=TIGHT_UM, loose_um=LOOSE_UM, vel_blend=VEL_BLEND):
    def hun(P, C, pred, pi, ci, gate):
        if len(pi) == 0 or len(ci) == 0: return []
        BIG = 1e9
        Draw = np.sqrt(((P[pi][:, None]-C[ci][None])**2).sum(2))
        D = np.sqrt(((pred[pi][:, None]-C[ci][None])**2).sum(2))
        cost = np.where(Draw > gate, BIG, D)
        ri, rc = linear_sum_assignment(cost)
        return [(int(pi[r]), int(ci[c])) for r, c in zip(ri, rc) if cost[r, c] < BIG]
    edges = []; prev_xyz = np.zeros((0, 3)); prev_ids = []; prev_vel = None
    for t in range(len(frames)):
        curr = np.asarray(frames[t], float).reshape(-1, 3); curr_ids = fid[t]
        if t > 0 and len(prev_ids) and len(curr):
            P = prev_xyz*VOXEL[None, :]; C = curr*VOXEL[None, :]
            pred = P + (vel_blend*prev_vel if prev_vel is not None else 0.0)
            N, M = len(P), len(C)
            links = hun(P, C, pred, np.arange(N), np.arange(M), min(tight_um, loose_um))
            up = {p for p, _ in links}; uc = {c for _, c in links}
            fp = np.array([i for i in range(N) if i not in up], int)
            fc = np.array([j for j in range(M) if j not in uc], int)
            links += hun(P, C, pred, fp, fc, loose_um)
            vel = np.zeros((N, 3)); nv = np.zeros((M, 3))
            for p, c in links:
                edges.append((prev_ids[p], curr_ids[c])); vel[p] = (curr[c]-prev_xyz[p])*VOXEL
            for p, c in links: nv[c] = vel[p]
            prev_vel = nv
        else:
            prev_vel = None
        prev_xyz, prev_ids = curr, curr_ids
    return edges

def close_gaps(frames, fid, nodes, edges, max_gap=MAX_GAP, gap_dist_um=GAP_DIST_UM):
    if not edges: return nodes, edges
    coords = {n[0]: (n[1], n[2], n[3], n[4]) for n in nodes}
    has_out = set(s for s, _ in edges); has_in = set(t for _, t in edges)
    ends, starts = {}, {}
    for nid, (t, z, y, x) in coords.items():
        if nid not in has_out: ends.setdefault(t, []).append(nid)
        if nid not in has_in: starts.setdefault(t, []).append(nid)
    new_nodes = list(nodes); new_edges = list(edges); nxt = max(coords) + 1
    for gap in range(1, max_gap+1):
        for t, es in ends.items():
            ss = starts.get(t+gap+1)
            if not ss: continue
            ec = np.array([[coords[e][1], coords[e][2], coords[e][3]] for e in es])*VOXEL
            sc = np.array([[coords[s][1], coords[s][2], coords[s][3]] for s in ss])*VOXEL
            d = np.sqrt(((ec[:, None, :]-sc[None, :, :])**2).sum(2)); thr = gap_dist_um*(gap+1); big = thr*1000+1
            cost = np.where(d <= thr, d, big); ri, ci = linear_sum_assignment(cost); used = set()
            for r, c in zip(ri, ci):
                if d[r, c] > thr or es[r] in has_out or ss[c] in used: continue
                e_id, s_id = es[r], ss[c]; te, ze, ye, xe = coords[e_id]; ts, zs, yy2, xs = coords[s_id]; prev = e_id
                for k in range(1, gap+1):
                    fr = k/(gap+1); nid = nxt; nxt += 1
                    new_nodes.append((nid, te+k, ze+(zs-ze)*fr, ye+(yy2-ye)*fr, xe+(xs-xe)*fr))
                    new_edges.append((prev, nid)); prev = nid
                new_edges.append((prev, s_id)); has_out.add(e_id); used.add(c)
    return new_nodes, new_edges

def filter_short(nodes, edges, min_len=FILTER_MIN_LEN):
    if not edges or min_len <= 1: return nodes, edges
    par = {n[0]: n[0] for n in nodes}
    def find(a):
        while par[a] != a: par[a] = par[par[a]]; a = par[a]
        return a
    for s, t in edges:
        ra, rb = find(s), find(t)
        if ra != rb: par[ra] = rb
    comp = defaultdict(list)
    for n in nodes: comp[find(n[0])].append(n[0])
    keep = set()
    for m in comp.values():
        if len(m) >= min_len: keep.update(m)
    nn = [n for n in nodes if n[0] in keep]
    ee = [(s, t) for s, t in edges if s in keep and t in keep]
    return nn, ee

def v2_link(frames):
    nodes, fid = build_nodes(frames)
    edges = link_twopass(frames, fid)
    nodes, edges = close_gaps(frames, fid, nodes, edges)
    # prune isolated
    used = set();
    for s, t in edges: used.add(s); used.add(t)
    nodes = [n for n in nodes if n[0] in used]
    nodes, edges = filter_short(nodes, edges)
    pred_nodes = [(n[0], n[1], int(round(n[2])), int(round(n[3])), int(round(n[4]))) for n in nodes]
    return pred_nodes, edges


# ---------------- tracksdata GLOBAL ILP linking (replaces frame-to-frame Hungarian) ----------------
# candidate edges are generated in PHYSICAL (um) space; only the linker differs from v2 (detection identical).
LINK_GATE_UM = 8.0      # consecutive-frame detections within this um are candidate links (== v2 loose gate)
MAX_NEIGH    = 5        # cap candidate children per node -> bounds the ILP size on dense frames
EDGE_BIAS_UM = 8.0      # edge weight w = dist_um - bias  (links shorter than bias get NEGATIVE weight = reward)
APPEAR_W     = 0.1      # cost to start a track
DISAPPEAR_W  = 0.1      # cost to end a track
DIVISION_W   = 1000.0   # HIGH -> effectively strict 1:1 (clean global-vs-greedy A/B). Lower this to enable divisions.

def _read_solution_edges(sol):
    # primary: node_ids() + successors() (documented to return list[int] of successors)
    try:
        src_l, tgt_l = [], []
        for n in sol.node_ids():
            succ = sol.successors(n)
            if isinstance(succ, dict):
                succ = succ.get(n, [])
            for m in (succ or []):
                src_l.append(int(n)); tgt_l.append(int(m))
        return src_l, tgt_l
    except Exception as e:
        print('   successors readback failed:', repr(e))
    # fallback: edge_attrs polars frame with source/target columns
    df = sol.edge_attrs()
    cols = list(df.columns)
    sk = 'source_id' if 'source_id' in cols else ('source' if 'source' in cols else cols[0])
    tk = 'target_id' if 'target_id' in cols else ('target' if 'target' in cols else cols[1])
    return df[sk].to_list(), df[tk].to_list()

def tracksdata_ilp_link(frames):
    import tracksdata as td
    from scipy.spatial import cKDTree
    g = td.graph.InMemoryGraph()
    import polars as pl
    # tracksdata validates attr keys; only 't' is built-in. We DON'T need z/y/x on the tracksdata graph
    # (candidate edges are computed by cKDTree on `frames`; the ILP uses only edge weight 'w' + time 't').
    if 'w' not in g.edge_attr_keys():
        g.add_edge_attr_key('w', pl.Float64(), 0.0)
    td2our = {}; our_nodes = []; frame_ids = []; nid = 1
    for t, coords in enumerate(frames):
        row = []
        for (z, y, x) in coords:
            tid = g.add_node({'t': int(t)})
            td2our[tid] = nid
            our_nodes.append((nid, t, int(round(z)), int(round(y)), int(round(x))))
            row.append((tid, np.array([z, y, x], float))); nid += 1
        frame_ids.append(row)
    n_edge = 0
    for t in range(len(frames) - 1):
        A, B = frame_ids[t], frame_ids[t + 1]
        if not A or not B: continue
        Bxyz = np.array([b[1] for b in B]) * VOXEL
        Axyz = np.array([a[1] for a in A]) * VOXEL
        tree = cKDTree(Bxyz); k = min(MAX_NEIGH, len(B))
        for ai, (a_id, _) in enumerate(A):
            dist, idx = tree.query(Axyz[ai], k=k)
            for d, bi in zip(np.atleast_1d(dist), np.atleast_1d(idx)):
                if d <= LINK_GATE_UM:
                    g.add_edge(int(a_id), int(B[int(bi)][0]), {'w': float(d) - EDGE_BIAS_UM})
                    n_edge += 1
    solver = td.solvers.ILPSolver(edge_weight='w', appearance_weight=APPEAR_W,
                                  disappearance_weight=DISAPPEAR_W, division_weight=DIVISION_W)
    sol = solver.solve(g)
    if sol is None:
        print('   ILP returned None'); return our_nodes, []
    src, tgt = _read_solution_edges(sol)
    pred_edges = [(td2our[s], td2our[t_]) for s, t_ in zip(src, tgt) if s in td2our and t_ in td2our]
    print(f'   ILP graph {g.num_nodes()}n/{n_edge}e -> solution {len(pred_edges)} edges')
    return our_nodes, pred_edges

# ---------------- main ----------------
wpath = None
for pat in ('v1_UNet_best.pt', 'unet_heatmap.pt', 'unet_latest.pt'):
    hits = [h for h in sorted(glob.glob('/kaggle/input/**/' + pat, recursive=True)) if 'trackastra' not in h.lower()]
    if hits: wpath = hits[0]; break
assert wpath, 'no v1 weights found under /kaggle/input (attach biohub-v1-unet-base16-heatmap)'
print('weights:', wpath)
model = UNet3D().to(device)
ck = torch.load(wpath, map_location=device)
model.load_state_dict(ck.get('best_model') or ck.get('model') or ck if isinstance(ck, dict) else ck)
model.eval()

samples = sample_list(TRAIN_DIR)[-N_VAL:]
print('eval samples:', [os.path.basename(g)[:-5] for _, g in samples])
print(f'ILP cfg: gate={LINK_GATE_UM}um maxneigh={MAX_NEIGH} bias={EDGE_BIAS_UM} appear={APPEAR_W} disappear={DISAPPEAR_W} div={DIVISION_W}')

agg = {'v2': [0, 0, 0], 'ilp': [0, 0, 0]}
div_tot = {'TP': 0, 'FP': 0, 'FN': 0}; div_pred_total = 0
for zp, gp in samples:
    t0 = time.time(); name = os.path.basename(gp)[:-5]
    arr = open_image(zp)
    gt_nodes, gt_edges = gt_nodes_edges(gp)
    frames = detect(model, arr)
    npred = sum(len(f) for f in frames)
    v2n, v2e = v2_link(frames)
    rv2, _ = edge_jaccard(gt_nodes, gt_edges, v2n, v2e)
    try:
        iln, ile = tracksdata_ilp_link(frames)
        ril, g2p = edge_jaccard(gt_nodes, gt_edges, iln, ile)
        rdiv = division_jaccard(gt_nodes, gt_edges, iln, ile, g2p)
    except Exception as e:
        import traceback; traceback.print_exc()
        print('   ILP FAILED on', name, ':', repr(e)[:300])
        ril = dict(J=0, TP=0, FP=0, FN=len(gt_edges)); rdiv = dict(divJ=0, TP=0, FP=0, FN=0, n_gt_div=0, n_pred_div=0)
    for k, r in (('v2', rv2), ('ilp', ril)):
        agg[k][0] += r['TP']; agg[k][1] += r['FP']; agg[k][2] += r['FN']
    div_tot['TP'] += rdiv['TP']; div_tot['FP'] += rdiv['FP']; div_tot['FN'] += rdiv['FN']; div_pred_total += rdiv['n_pred_div']
    print(f"{name}: preds={npred} | v2 J={rv2['J']:.3f} (TP{rv2['TP']} FP{rv2['FP']} FN{rv2['FN']}) | "
          f"ilp J={ril['J']:.3f} (TP{ril['TP']} FP{ril['FP']} FN{ril['FN']}) | "
          f"div TP{rdiv['TP']} FP{rdiv['FP']} FN{rdiv['FN']} nPredDiv{rdiv['n_pred_div']} ({time.time()-t0:.0f}s)")

def micro(a):
    TP, FP, FN = a; d = TP + FP + FN; return TP / d if d else 0.0

print()
print('==== MICRO edge-Jaccard (5 val, SAME detections; ONLY the linker differs) ====')
print(f"  v2  two-pass (greedy) : {micro(agg['v2']):.4f}  (TP{agg['v2'][0]} FP{agg['v2'][1]} FN{agg['v2'][2]})   [reference 0.8594]")
print(f"  tracksdata GLOBAL ILP : {micro(agg['ilp']):.4f}  (TP{agg['ilp'][0]} FP{agg['ilp'][1]} FN{agg['ilp'][2]})")
dd = div_tot['TP'] + div_tot['FP'] + div_tot['FN']
print(f"  ILP division-J: {(div_tot['TP']/dd if dd else 0.0):.4f}  (TP{div_tot['TP']} FP{div_tot['FP']} FN{div_tot['FN']}, nPredDiv total {div_pred_total})")
print()
print('READ: ILP edge-J >= v2 0.8594 -> global linking beats greedy on identical detections -> paradigm has legs.')
print('      then lower DIVISION_W to add divisions (watch nPredDiv for a trackastra-style flood).')


Writing /tmp/v7_ilp_eval.py


In [3]:
import subprocess, sys, os, time
env = dict(os.environ, POLARS_PREFER_PKG='32')
t0 = time.time()
r = subprocess.run([sys.executable, '/tmp/v7_ilp_eval.py'], capture_output=True, text=True, env=env)
print(r.stdout[-16000:]); print('--- STDERR tail ---'); print(r.stderr[-3000:])
print('elapsed', round(time.time() - t0), 's | returncode', r.returncode)


device cuda | numpy 2.0.2 | torch 2.10.0+cu128
weights: /kaggle/input/models/lingxd/biohub-v1-unet-base16-heatmap/pytorch/default/1/v1_UNet_best.pt
eval samples: ['6bba_fbc898dc', '6bba_fc516dc6', '6bba_fc5f39dc', '6bba_fc83837d', '6bba_fe670320']
ILP cfg: gate=8.0um maxneigh=5 bias=8.0 appear=0.1 disappear=0.1 div=1000.0
[07/10/26 04:56:47] WARNING  Solver failed with Gurobi,       _ilp_solver.py:363
                             trying Scip.                                       
                             Got error:                                         
                             Gurobi license is not available.                   
   ILP graph 4896n/5333e -> solution 4485 edges
6bba_fbc898dc: preds=4896 | v2 J=0.961 (TP614 FP0 FN25) | ilp J=0.961 (TP614 FP0 FN25) | div TP0 FP0 FN0 nPredDiv0 (63s)
[07/10/26 04:57:42] WARNING  Solver failed with Gurobi,       _ilp_solver.py:363
                             trying Scip.                                       
                     